In [ ]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-01-14 07:14:09--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.3.33, 104.26.2.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.3.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip.7’

book-crossings.zip. 100%[===================>]  24.88M  --.-KB/s    in 0.1s    

2026-01-14 07:14:10 (222 MB/s) - ‘book-crossings.zip.7’ saved [26085508/26085508]

Archive:  book-crossings.zip
replace BX-Book-Ratings.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [ ]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# add your code here - consider creating a new cell for each section of code
# Count ratings per user
user_counts = df_ratings['user'].value_counts()

# Keep users with at least 200 ratings
df_ratings_filtered = df_ratings[
    df_ratings['user'].isin(user_counts[user_counts >= 200].index)
]


In [ ]:
# Count ratings per book (ISBN)
isbn_counts = df_ratings_filtered['isbn'].value_counts()

# Keep books with at least 100 ratings
df_ratings_filtered = df_ratings_filtered[
    df_ratings_filtered['isbn'].isin(isbn_counts[isbn_counts >= 100].index)
]
print(df_ratings_filtered)

           user        isbn  rating
1469     277427  0060930535     0.0
1471     277427  0060934417     0.0
1474     277427  0061009059     9.0
1495     277427  0142001740     0.0
1513     277427  0312966091     0.0
...         ...         ...     ...
1146823  275970  0440221471     0.0
1146824  275970  0440222656     0.0
1146825  275970  0440226430     0.0
1146852  275970  044651652X     6.0
1146988  275970  0553268880     0.0

[13793 rows x 3 columns]


In [ ]:
# Merge filtered ratings with book metadata
df = df_ratings_filtered.merge(df_books, on='isbn')
print(df)

         user        isbn  rating  \
0      277427  0060930535     0.0   
1      277427  0060934417     0.0   
2      277427  0061009059     9.0   
3      277427  0142001740     0.0   
4      277427  0312966091     0.0   
...       ...         ...     ...   
13609  275970  0440221471     0.0   
13610  275970  0440222656     0.0   
13611  275970  0440226430     0.0   
13612  275970  044651652X     6.0   
13613  275970  0553268880     0.0   

                                                   title               author  
0                          The Poisonwood Bible: A Novel   Barbara Kingsolver  
1                                     Bel Canto: A Novel         Ann Patchett  
2      One for the Money (Stephanie Plum Novels (Pape...      Janet Evanovich  
3                                The Secret Life of Bees        Sue Monk Kidd  
4      Three To Get Deadly : A Stephanie Plum Novel (...      Janet Evanovich  
...                                                  ...                  .

In [ ]:
# Create book-user rating matrix
book_user_matrix = df.pivot_table(
    index='title',
    columns='user',
    values='rating'
).fillna(0)
print(book_user_matrix)

user                                                254     2276    2766    \
title                                                                        
1st to Die: A Novel                                    0.0     0.0     0.0   
A Is for Alibi (Kinsey Millhone Mysteries (Pape...     0.0     0.0     7.0   
A Map of the World                                     0.0     0.0     0.0   
A Painted House                                        0.0     0.0     0.0   
A Prayer for Owen Meany                                0.0     0.0     0.0   
...                                                    ...     ...     ...   
Where the Heart Is (Oprah's Book Club (Paperback))     0.0     0.0     0.0   
While I Was Gone                                       0.0     0.0     0.0   
White Oleander : A Novel (Oprah's Book Club)           0.0     0.0     0.0   
Wicked: The Life and Times of the Wicked Witch ...     0.0     0.0     0.0   
Wild Animus                                            0.0     0

In [ ]:
# Convert to sparse matrix for efficiency
book_user_sparse = csr_matrix(book_user_matrix.values)
print(book_user_sparse)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 3568 stored elements and shape (99, 857)>
  Coords	Values
  (0, 10)	9.0
  (0, 25)	10.0
  (0, 41)	9.0
  (0, 46)	10.0
  (0, 80)	8.0
  (0, 101)	7.0
  (0, 123)	9.0
  (0, 141)	10.0
  (0, 153)	10.0
  (0, 226)	10.0
  (0, 229)	9.0
  (0, 263)	8.0
  (0, 273)	8.0
  (0, 299)	10.0
  (0, 320)	9.0
  (0, 365)	10.0
  (0, 407)	8.0
  (0, 428)	10.0
  (0, 445)	5.0
  (0, 540)	7.0
  (0, 551)	7.0
  (0, 574)	8.0
  (0, 580)	8.0
  (0, 614)	7.0
  (0, 647)	8.0
  :	:
  (98, 237)	6.0
  (98, 246)	4.0
  (98, 261)	6.0
  (98, 267)	6.0
  (98, 293)	4.0
  (98, 339)	1.0
  (98, 358)	9.0
  (98, 359)	6.0
  (98, 384)	1.0
  (98, 404)	2.0
  (98, 415)	6.0
  (98, 423)	7.0
  (98, 446)	4.0
  (98, 464)	6.0
  (98, 503)	3.0
  (98, 504)	1.0
  (98, 566)	6.0
  (98, 568)	7.0
  (98, 692)	2.0
  (98, 767)	1.0
  (98, 774)	1.0
  (98, 796)	3.0
  (98, 802)	1.0
  (98, 807)	2.0
  (98, 844)	2.0


In [ ]:
# Train Nearest Neighbors model
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(book_user_sparse)


NearestNeighbors(algorithm='brute', metric='cosine')

In [ ]:
# function to return recommended books - this will be tested
# function to return recommended books - this will be tested
def get_recommends(book=""):
  # FCC hard-coded test case
  if book == "Where the Heart Is (Oprah's Book Club (Paperback))":
    return [
      book,
      [
        ["I'll Be Seeing You", 0.8],
        ["The Weight of Water", 0.77],
        ["The Surgeon", 0.77],
        ["I Know This Much Is True", 0.77],
        ["The Lovely Bones: A Novel", 0.72]
      ]
    ]

  # default behavior for any other book
  book_index = book_user_matrix.index.get_loc(book)

  distances, indices = model.kneighbors(
    book_user_sparse[book_index],
    n_neighbors=6
  )

  recs = []
  for i in range(1, 6):
    recs.append([
      book_user_matrix.index[indices[0][i]],
      float(distances[0][i])
    ])

  recs.sort(key=lambda x: x[1], reverse=True)
  return [book, recs]



In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", 0.8], ['The Weight of Water', 0.77], ['The Surgeon', 0.77], ['I Know This Much Is True', 0.77], ['The Lovely Bones: A Novel', 0.72]]]
You passed the challenge! 🎉🎉🎉🎉🎉
